# Bird's Eye View (BEV) Visualization

This notebook walks through how modern **BEV perception** stacks work, with a focus on the **Lift-Splat-Shoot (LSS)** technique.

**What you will see:**
1. Load a pretrained BEV network (LSS) and run it on a sample surround-camera scene.
2. Visualize the *lift* step — the same feature map before and after being lifted into 3D.
3. Inspect the learned depth distribution, its expected value, and its uncertainty.
4. (Later parts) LiDAR-camera fusion, occupancy, and planning — all in the BEV grid.

> Reference: *Lift, Splat, Shoot: Encoding Images From Arbitrary Camera Rigs by Implicitly Unprojecting to 3D* — Philion & Fidler, ECCV 2020.

## Part 1 — Load a pretrained BEV Network (LSS)

We will:

1. Install dependencies and clone the official LSS repo.
2. Download the pretrained weights released by NVIDIA.
3. Build the `LiftSplatShoot` model and load the checkpoint.
4. Load a sample surround-camera input (6 cameras) and run the network.

> **Note:** the original LSS code assumes a specific directory layout. We mirror the inference setup from [`src/explore.py`](https://github.com/nv-tlabs/lift-splat-shoot/blob/master/src/explore.py) so that we can reuse NVIDIA's weights directly.

### 1.1 — Install dependencies

In [ ]:
!pip install pyquaternion
!pip install nuscenes-devkit tensorboardX efficientnet_pytorch==0.7.0

# Clone the official Lift-Splat-Shoot repo (NVIDIA Toronto AI Lab)
![ -d lift-splat-shoot ] || git clone --quiet https://github.com/nv-tlabs/lift-splat-shoot.git

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('lift-splat-shoot'))
print('LSS sources available at:', os.path.abspath('lift-splat-shoot'))

LSS sources available at: /content/lift-splat-shoot


### 1.2 — Download the pretrained LSS weights

NVIDIA hosts the checkpoint on Google Drive (see the LSS README). We fetch it with `gdown`. The file is ~200 MB and only needs to be downloaded once.

In [2]:
import os, gdown

WEIGHTS_PATH = 'model525000.pt'
LSS_WEIGHTS_URL = 'https://drive.google.com/uc?id=18fy-6beTFTZx5SrYLs9Xk7cY-fGSm7kw'

if not os.path.exists(WEIGHTS_PATH):
    try:
        gdown.download(LSS_WEIGHTS_URL, WEIGHTS_PATH, quiet=False)
    except Exception as e:
        print('Could not download pretrained weights:', e)
        print('Falling back to an ImageNet-initialised backbone later.')

print('Weights available:', os.path.exists(WEIGHTS_PATH))

Weights available: True


### 1.3 — Build the LSS model and load the checkpoint

The LSS model is built from two main pieces:

- `CamEncode` — an EfficientNet-B0 backbone that, for every pixel, predicts both a **feature vector** and a **categorical depth distribution** over a set of depth bins.
- `BevEncode` — a small ResNet that cleans up the rasterised BEV feature grid after the splat step.

In [3]:
import torch
from src.models import compile_model  # from the LSS repo

# These hyper-parameters match the settings NVIDIA used to train model525000.pt
grid_conf = {
    'xbound': [-50.0, 50.0, 0.5],   # BEV grid: x in [-50, 50] m, 0.5 m per cell
    'ybound': [-50.0, 50.0, 0.5],
    'zbound': [-10.0, 10.0, 20.0],
    'dbound': [4.0, 45.0, 1.0],     # 41 depth bins from 4 m to 45 m
}
data_aug_conf = {
    'resize_lim': (0.193, 0.225),
    'final_dim': (128, 352),        # network input size per camera
    'rot_lim': (-5.4, 5.4),
    'H': 900, 'W': 1600,
    'rand_flip': True,
    'bot_pct_lim': (0.0, 0.22),
    'cams': ['CAM_FRONT_LEFT', 'CAM_FRONT', 'CAM_FRONT_RIGHT',
             'CAM_BACK_LEFT',  'CAM_BACK',  'CAM_BACK_RIGHT'],
    'Ncams': 6,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = compile_model(grid_conf, data_aug_conf, outC=1)  # outC=1 => drivable-area head

if os.path.exists(WEIGHTS_PATH):
    state = torch.load(WEIGHTS_PATH, map_location='cpu')
    model.load_state_dict(state)
    print('Loaded pretrained LSS weights.')
else:
    print('Using randomly initialised LSS (pretrained weights unavailable).')

model = model.to(device).eval()
n_params = sum(p.numel() for p in model.parameters())
print(f'Model on {device} — {n_params/1e6:.1f} M parameters')

Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


100%|██████████| 20.4M/20.4M [00:00<00:00, 102MB/s] 
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Loaded pretrained weights for efficientnet-b0
Loaded pretrained LSS weights.
Model on cuda — 14.3 M parameters


### 1.4 — Load a sample surround-camera input

LSS expects, for every one of the 6 cameras, a tensor of shape `(3, 128, 352)` together with the camera intrinsics and the camera → ego-vehicle extrinsics.

To keep the notebook self-contained we use the nuScenes **mini** split via `nuscenes-devkit` when it is available, and otherwise fall back to a synthetic scene with the canonical nuScenes camera rig. Either way the tensor shapes fed into the network are identical.

In [4]:
import numpy as np
from PIL import Image
import torchvision.transforms.functional as TF

CAMS = data_aug_conf['cams']
H, W = data_aug_conf['final_dim']  # 128 x 352

def synth_surround_scene(seed=0):
    """Generate a toy 6-camera scene so the notebook always runs.
    Each camera gets a sky/ground split with a few coloured 'vehicles'.
    Returns a list of PIL.Image of size (W, H)."""
    rng = np.random.default_rng(seed)
    imgs = []
    for i, cam in enumerate(CAMS):
        img = np.zeros((H, W, 3), dtype=np.uint8)
        # Sky gradient
        for y in range(H // 2):
            img[y] = (135 - y//2, 180 - y//3, 235)
        # Road
        img[H//2:] = (70, 70, 75)
        # Lane markings
        for lane_x in (W//3, 2*W//3):
            for y in range(H//2, H, 8):
                img[y:y+3, lane_x-1:lane_x+1] = 230
        # Random vehicles
        for _ in range(rng.integers(1, 4)):
            cx, cy = rng.integers(20, W-20), rng.integers(H//2+5, H-15)
            w, h = rng.integers(20, 50), rng.integers(10, 22)
            color = rng.integers(40, 220, size=3)
            img[max(0,cy-h):cy, max(0,cx-w//2):cx+w//2] = color
        imgs.append(Image.fromarray(img))
    return imgs

pil_images = synth_surround_scene(seed=7)

def pil_to_tensor(pil_img):
    # Same normalisation as the original LSS dataloader (src/data.py)
    t = TF.to_tensor(pil_img)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (t - mean) / std

imgs = torch.stack([pil_to_tensor(im) for im in pil_images])  # (6, 3, H, W)
print('imgs', tuple(imgs.shape))

imgs (6, 3, 128, 352)


In [5]:
# Canonical nuScenes camera rig (rough values — good enough for visualisation).
# Yaw angles (deg) of each camera around the ego-Z axis, and their (x, y, z) offsets (m) in ego frame.
RIG = {
    'CAM_FRONT_LEFT':  (dict(yaw= 55.0, xyz=( 1.52,  0.50, 1.50))),
    'CAM_FRONT':       (dict(yaw=  0.0, xyz=( 1.72,  0.00, 1.50))),
    'CAM_FRONT_RIGHT': (dict(yaw=-55.0, xyz=( 1.52, -0.50, 1.50))),
    'CAM_BACK_LEFT':   (dict(yaw=110.0, xyz=( 1.04,  0.48, 1.50))),
    'CAM_BACK':        (dict(yaw=180.0, xyz=( 0.05,  0.00, 1.50))),
    'CAM_BACK_RIGHT':  (dict(yaw=-110.0,xyz=( 1.04, -0.48, 1.50))),
}

def intrinsics(fx=1266.0, fy=1266.0, cx=816.0, cy=491.0):
    K = torch.tensor([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=torch.float32)
    return K

def yaw_matrix(deg):
    a = np.deg2rad(deg)
    c, s = np.cos(a), np.sin(a)
    return torch.tensor([[c, -s, 0], [s, c, 0], [0, 0, 1]], dtype=torch.float32)

# Build the tensors LSS expects: intrins, rots, trans, post_rots, post_trans
intrins, rots, trans = [], [], []
for cam in CAMS:
    intrins.append(intrinsics())
    rots.append(yaw_matrix(RIG[cam]['yaw']))
    trans.append(torch.tensor(RIG[cam]['xyz'], dtype=torch.float32))
intrins = torch.stack(intrins)
rots    = torch.stack(rots)
trans   = torch.stack(trans)

# Post-augmentation transforms: identity (we already resized/normalised the images).
post_rots  = torch.eye(3).unsqueeze(0).repeat(6, 1, 1)
post_trans = torch.zeros(6, 3)

# Add the batch dimension expected by the model
batch = [x.unsqueeze(0).to(device) for x in (imgs, rots, trans, intrins, post_rots, post_trans)]
for name, t in zip(['imgs','rots','trans','intrins','post_rots','post_trans'], batch):
    print(f'{name:11s} {tuple(t.shape)}')

imgs        (1, 6, 3, 128, 352)
rots        (1, 6, 3, 3)
trans       (1, 6, 3)
intrins     (1, 6, 3, 3)
post_rots   (1, 6, 3, 3)
post_trans  (1, 6, 3)


### 1.4b — Swap in a **real** nuScenes surround sample

We now replace the toy images with one real keyframe (6 cameras) from the `v1.0-mini` split of nuScenes. The mini split is ~4 GB — it's the smallest public nuScenes bundle that ships with complete metadata/calibration. Download is a one-off; the tarball is cached and extraction is skipped on re-runs.

**What happens here:**
1. Download & extract `v1.0-mini.tgz` (skipped if already present — or if you manually upload the tarball to Colab).
2. Instantiate the official LSS `SegmentationData` loader — this guarantees the calibration (`rots`, `trans`, `intrins`, `post_rots`, `post_trans`) is **exactly** what the pretrained weights expect.
3. Overwrite the synthetic `pil_images` + `batch` tensors with the real sample. All downstream cells (visualize / Part 2) keep working as-is.

In [ ]:
import os, subprocess, sys

NUSCENES_ROOT = './nuscenes-mini'
TARBALL = '/tmp/v1.0-mini.tgz'

META_READY = os.path.exists(os.path.join(NUSCENES_ROOT, 'v1.0-mini', 'scene.json'))

if not META_READY:
    if not (os.path.exists(TARBALL) and os.path.getsize(TARBALL) > 10_000_000):
        # Try the official download URL. The nuScenes terms page has a direct
        # tarball link — if the URL ever changes, you can also upload the
        # tarball manually to /tmp/v1.0-mini.tgz and re-run this cell.
        CANDIDATE_URLS = [
            'https://www.nuscenes.org/data/v1.0-mini.tgz',
            'https://d36yt3mvayqw5m.cloudfront.net/public/v1.0/v1.0-mini.tgz',
            'https://motional-nuscenes.s3.amazonaws.com/public/v1.0/v1.0-mini.tgz',
        ]
        for url in CANDIDATE_URLS:
            print(f'Trying {url} …')
            rc = subprocess.call(['wget', '-q', '--show-progress', '-c', url, '-O', TARBALL])
            if rc == 0 and os.path.getsize(TARBALL) > 10_000_000:
                print('Downloaded from', url)
                break
        else:
            print('\nAuto-download failed. Please manually download v1.0-mini.tgz from')
            print('    https://www.nuscenes.org/download')
            print(f'and upload it to {TARBALL} in the Colab file tree, then re-run this cell.')

    if os.path.exists(TARBALL) and os.path.getsize(TARBALL) > 10_000_000:
        os.makedirs(NUSCENES_ROOT, exist_ok=True)
        print('Extracting …')
        subprocess.check_call(['tar', '-xzf', TARBALL, '-C', NUSCENES_ROOT])

META_READY = os.path.exists(os.path.join(NUSCENES_ROOT, 'v1.0-mini', 'scene.json'))
print('nuScenes metadata ready:', META_READY)


In [ ]:
# Use the official LSS dataloader to build one batch from nuScenes mini.
# This matches the exact preprocessing NVIDIA used when training model525000.pt.
from PIL import Image

if META_READY:
    from nuscenes.nuscenes import NuScenes
    from src.data import SegmentationData

    nusc = NuScenes(version='v1.0-mini', dataroot=NUSCENES_ROOT, verbose=False)
    # is_train=False => deterministic augmentation (center crop, no flip/rotate)
    ds = SegmentationData(nusc, is_train=False,
                          data_aug_conf=data_aug_conf, grid_conf=grid_conf)

    # Try other keyframes! The mini val split has ~80 samples across 2 scenes.
    # Good picks: 5 (intersection), 30 (straight road), 45 (urban), 70 (open lane).
    SAMPLE_IDX = 30
    imgs_r, rots_r, trans_r, intrins_r, post_rots_r, post_trans_r, binimg_r = ds[SAMPLE_IDX]

    # Convert the already-augmented model-input tensor back to a PIL image so
    # that overlays / scatter plots in later cells line up with the feature grid.
    _MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    _STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    def tensor_to_pil(t):
        x = (t.cpu() * _STD + _MEAN).clamp(0, 1)
        return Image.fromarray((x.permute(1, 2, 0).numpy() * 255).astype('uint8'))

    pil_images = [tensor_to_pil(imgs_r[i]) for i in range(6)]
    imgs  = imgs_r
    batch = [x.unsqueeze(0).to(device) for x in
             (imgs_r, rots_r, trans_r, intrins_r, post_rots_r, post_trans_r)]

    # Keep the ground-truth BEV mask around for later comparison.
    binimg_gt = binimg_r.cpu().numpy()

    rec = ds.ixes[SAMPLE_IDX]
    scene_name = nusc.get('scene', rec['scene_token'])['name']
    print(f'Loaded real nuScenes sample #{SAMPLE_IDX} from scene {scene_name!r}')
    for name, t in zip(['imgs','rots','trans','intrins','post_rots','post_trans'], batch):
        print(f'  {name:11s} {tuple(t.shape)}')
else:
    binimg_gt = None
    print('nuScenes mini not available — keeping the synthetic scene.')


### 1.5 — Run the network and visualize the inputs + BEV output

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# --- forward pass -------------------------------------------------------
with torch.no_grad():
    bev_logits = model(*batch)                       # (1, 1, 200, 200)
    bev = torch.sigmoid(bev_logits)[0, 0].cpu().numpy()

# --- nuScenes-style surround layout: cameras top & bottom, big BEV in the middle
fig = plt.figure(figsize=(14, 11))
gs  = GridSpec(3, 3, height_ratios=[1.0, 2.6, 1.0], hspace=0.18, wspace=0.05, figure=fig)

cam_positions = {
    'CAM_FRONT_LEFT':  (0, 0), 'CAM_FRONT':      (0, 1), 'CAM_FRONT_RIGHT': (0, 2),
    'CAM_BACK_LEFT':   (2, 0), 'CAM_BACK':       (2, 1), 'CAM_BACK_RIGHT':  (2, 2),
}
for cam, (r, c) in cam_positions.items():
    ax = fig.add_subplot(gs[r, c])
    ax.imshow(pil_images[CAMS.index(cam)])
    ax.set_title(cam, fontsize=9); ax.axis('off')

# Big central BEV plot (forward up, ego-left on the left)
ax_bev = fig.add_subplot(gs[1, :])
im = ax_bev.imshow(bev, origin='lower', cmap='magma', extent=[-50, 50, -50, 50], vmin=0, vmax=1)
ax_bev.invert_xaxis()
ax_bev.set_aspect('equal')

# --- annotations that make the BEV easier to read ------------------------
# Distance reference rings around the ego (10, 20, 30, 40 m)
for r in [10, 20, 30, 40]:
    circle = plt.Circle((0, 0), r, fill=False, color='white', alpha=0.28,
                        linestyle=':', lw=1.0)
    ax_bev.add_patch(circle)
    ax_bev.text(0, r + 0.8, f'{r} m', color='white', fontsize=8, alpha=0.7,
                ha='center', va='bottom')

# Forward-direction arrow from the ego
ax_bev.annotate('', xy=(0, 14), xytext=(0, 1.5),
                arrowprops=dict(arrowstyle='-|>', color='cyan', lw=2.8, alpha=0.95,
                                mutation_scale=22))
ax_bev.text(0, 16, 'FORWARD', color='cyan', fontsize=10, fontweight='bold',
            ha='center', va='bottom')

ax_bev.scatter([0], [0], c='cyan', marker='^', s=320, edgecolor='black',
               linewidth=1.8, zorder=6)

ax_bev.set_xlabel('y  [m]   (+ = left of ego)', fontsize=11)
ax_bev.set_ylabel('x  [m]   (+ = forward)',     fontsize=11)
ax_bev.set_title(
    'LSS drivable-area output — for every 0.5 m × 0.5 m cell in a 100 m × 100 m box around the ego,
'
    '"what is the probability that this cell is drivable road?"   '
    'Bright = yes,  dark = no.',
    fontsize=11, pad=10)
plt.colorbar(im, ax=ax_bev, fraction=0.025, pad=0.01, label='P(drivable)')

fig.suptitle('Part 1 — 6 surround cameras  →  Lift-Splat-Shoot  →  one top-down drivable map',
             fontsize=13, fontweight='bold', y=0.995)
plt.show()

print(f'BEV tensor: shape={bev.shape}, range=[{bev.min():.3f}, {bev.max():.3f}]')


## Part 2 — Visualizing the **Lift** step

> **Heads up on the input data.** The previous cells use a synthetic surround-camera scene so the notebook can run anywhere. NVIDIA's pretrained LSS weights *are* loaded, but the depth-net has never seen toy images like this, so the predicted depths/BEV mask are not meaningful. The *mechanics* of the lift, however, are identical — which is exactly what we want to visualize here. To swap in a real nuScenes sample, install `nuscenes-devkit`, download the `v1.0-mini` split, and replace the `pil_images` / `rots` / `trans` / `intrins` tensors with the ones returned by `NuscData.__getitem__` (see `lift-splat-shoot/src/data.py`).

The lift turns every pixel feature into a *ray of features* — one per depth bin — weighted by a learned categorical distribution over depth. Concretely:

```
for each pixel (u, v):
    f(u, v)      ∈ R^C            # context feature vector
    α(u, v, d)   ∈ Δ^D            # softmax over D depth bins
    lifted(u, v, d) = α(u, v, d) · f(u, v)    ∈ R^C
```

We now extract these intermediate tensors from the pretrained `CamEncode` module.

In [ ]:
# Extract intermediate tensors from the CamEncode module for all 6 cameras
imgs_b, rots_b, trans_b, intrins_b, post_rots_b, post_trans_b = batch
B, N, _, imH, imW = imgs_b.shape
imgs_flat = imgs_b.view(B * N, 3, imH, imW)

with torch.no_grad():
    eff_feat     = model.camencode.get_eff_depth(imgs_flat)   # (N, 512, fH, fW)
    depth_logits = model.camencode.depthnet(eff_feat)         # (N, D+C, fH, fW)
    D_ = model.camencode.D
    C_ = model.camencode.C
    depth_dist = depth_logits[:, :D_].softmax(dim=1)          # (N, D, fH, fW)
    ctx_feat   = depth_logits[:, D_:]                         # (N, C, fH, fW)
    lifted     = depth_dist.unsqueeze(1) * ctx_feat.unsqueeze(2)  # (N, C, D, fH, fW)

fH, fW = eff_feat.shape[-2:]
print(f'Feature grid per camera : {fH} x {fW}   (input {imH}x{imW}, stride {imH//fH})')
print(f'Depth bins              : D = {D_}  from {grid_conf["dbound"][0]} m to {grid_conf["dbound"][1]} m')
print(f'Context feature channels: C = {C_}')
print('')
print(f'2D features before lift : ctx_feat    {tuple(ctx_feat.shape)}')
print(f'Depth distribution      : depth_dist  {tuple(depth_dist.shape)}')
print(f'Lifted 3D frustum feats : lifted      {tuple(lifted.shape)}   # = depth_dist ⊗ ctx_feat')


### 2.1 — Features **before** the lift (2D feature maps)

The `CamEncode` module first runs an EfficientNet-B0 trunk and produces a dense feature map of shape `(C=64, fH=8, fW=22)` for each camera image. We project those 64-channel features down to 3 channels with PCA so we can visualize them as RGB.

Brighter / more saturated regions correspond to parts of the image the network finds informative.

In [ ]:
from sklearn.decomposition import PCA
from PIL import Image as PILImage

def feat_to_rgb(feat_2d):
    """(C, H, W) tensor -> (H, W, 3) RGB in [0, 1] via per-pixel PCA."""
    C_, H_, W_ = feat_2d.shape
    X = feat_2d.reshape(C_, -1).T.cpu().numpy()
    Y = PCA(n_components=3).fit_transform(X)
    Y = (Y - Y.min(0)) / (Y.max(0) - Y.min(0) + 1e-6)
    return Y.reshape(H_, W_, 3)

def upscale_rgb(rgb, out_H, out_W):
    """Bicubic-upscale an (H, W, 3) RGB array to (out_H, out_W, 3)."""
    img = PILImage.fromarray((np.clip(rgb, 0, 1) * 255).astype(np.uint8))
    return np.asarray(img.resize((out_W, out_H), PILImage.BICUBIC)) / 255.0

# Compute PCA features for every camera and upscale to the full image size so that
# spatial correspondence with the input is visible (the raw feature maps are only 8×22).
feat_rgbs = [upscale_rgb(feat_to_rgb(ctx_feat[i]), imH, imW) for i in range(len(CAMS))]

fig, axes = plt.subplots(2, 6, figsize=(17, 5.2))
for i, cam in enumerate(CAMS):
    axes[0, i].imshow(pil_images[i]);    axes[0, i].set_title(cam, fontsize=9); axes[0, i].axis('off')
    axes[1, i].imshow(feat_rgbs[i]);                                          axes[1, i].axis('off')

# Row labels on the far left
fig.text(0.012, 0.73, 'INPUT\nRGB image\n(what the\ncamera sees)',
         fontsize=10, ha='left', va='center',
         bbox=dict(boxstyle='round', facecolor='#d8ecff', edgecolor='#4a78b5'))
fig.text(0.012, 0.27, 'CNN\nFEATURES\n(64-d\u2192RGB\nvia PCA)',
         fontsize=10, ha='left', va='center',
         bbox=dict(boxstyle='round', facecolor='#efdfff', edgecolor='#7d4bb5'))

fig.suptitle("BEFORE the lift \u2014 what the 2D CNN 'sees'\n"
             "The EfficientNet backbone turns each 16\u202f\u00d7\u202f16 image region into a 64-d feature vector.\n"
             "PCA reduces those 64 numbers to 3 RGB channels, then we upscale back to image resolution.\n"
             "Same color = the CNN thinks those regions are semantically similar (road / car / building / sky / \u2026).",
             fontsize=10.5, y=1.09)
plt.tight_layout(rect=[0.06, 0, 1, 1])
plt.show()

print(f'Feature map shape per camera: {ctx_feat.shape[1]} channels \u00d7 {fH} \u00d7 {fW}   '
      f'(displayed upscaled to {imH} \u00d7 {imW})')

### 2.2 — Depth distribution per pixel — **interactive explorer**

For every cell of the downsampled feature map, LSS predicts a categorical distribution over `D = 41` depth bins (4 m … 45 m, step 1 m).

The cell below is **interactive**: slide the *row* / *col* knobs (or switch camera) to pick any feature cell — the depth PDF on the right redraws live.

**What to look for**

- *Sharp single peak*  →  the network is confident about depth.
- *Broad / bi-modal PDF*  →  the network is unsure (multiple plausible depths).
- Pixels on the **road surface** tend to peak near a specific depth.
- Pixels in the **sky** or on **very distant** objects typically show much broader distributions.


In [ ]:
# Interactive depth-distribution explorer
# ------------------------------------------------------------------------
# LSS predicts a full 41-bin probability distribution over depth for every
# 16x16 region of the image. Below, slide the row/column sliders (or change
# camera) to pick any one of those cells and see its distribution live.
# The red box on the left shows which 16x16 patch you're inspecting.

from matplotlib.patches import Rectangle

try:
    from ipywidgets import interact, IntSlider, Dropdown
except ImportError:
    !pip install -q ipywidgets
    from ipywidgets import interact, IntSlider, Dropdown

_depth_bins_np = depth_bins.cpu().numpy()
_stride_y, _stride_x = imH // fH, imW // fW

def _depth_explorer(cam='CAM_FRONT', fh=fH // 2, fw=fW // 2):
    c   = CAMS.index(cam)
    pdf = depth_dist[c, :, fh, fw].cpu().numpy()
    e_d = float((pdf * _depth_bins_np).sum())
    h_d = float(-(pdf * np.log(pdf + 1e-12)).sum())

    u_px = fw * _stride_x + _stride_x // 2
    v_px = fh * _stride_y + _stride_y // 2

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4.3))
    ax0.imshow(pil_images[c])
    ax0.add_patch(Rectangle((fw * _stride_x, fh * _stride_y), _stride_x, _stride_y,
                            linewidth=2.5, edgecolor='red', facecolor='none'))
    ax0.scatter([u_px], [v_px], c='red', s=80, edgecolor='white', linewidths=2, zorder=5)
    ax0.set_title(f'{cam}  —  selected feature cell (row={fh}, col={fw})  ≈  image pixel ({v_px}, {u_px})',
                  fontsize=10)
    ax0.axis('off')

    ax1.bar(_depth_bins_np, pdf, width=0.9, color='steelblue', edgecolor='navy', alpha=0.8)
    ax1.axvline(e_d, color='limegreen', lw=2.2, label=f'E[d] = {e_d:.1f} m')
    ax1.set_xlabel('depth bin  [m]'); ax1.set_ylabel('P(depth)')
    ax1.set_title(f'Depth distribution at this cell   (entropy = {h_d:.2f} nats)', fontsize=10)
    ax1.set_xlim(3.5, 45.5)
    ax1.set_ylim(0, max(pdf.max() * 1.15, 0.05))
    ax1.legend(); ax1.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

print(f'Feature grid per camera: {fH} rows × {fW} cols   (each cell ↔ a {_stride_y} × {_stride_x} image patch)')
interact(
    _depth_explorer,
    cam=Dropdown(options=CAMS, value='CAM_FRONT', description='camera'),
    fh=IntSlider(min=0, max=fH - 1, step=1, value=fH // 2, description='row (fh)'),
    fw=IntSlider(min=0, max=fW - 1, step=1, value=fW // 2, description='col (fw)'),
);


### 2.3 — A static snapshot: three pixels side-by-side

If you'd rather see a static picture (e.g. when the notebook is rendered on nbviewer where sliders don't work), here are the depth PDFs at three hand-picked pixels of the current CAM_FRONT frame.


In [ ]:
sample_pixels = [
    (fH // 4,     fW // 2),   # upper-middle (far)
    (fH // 2,     fW // 3),   # mid-left
    (3 * fH // 4, 2 * fW // 3),  # lower-right (close)
]

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4))
ax0.imshow(pil_images[cam_idx]); ax0.set_title(f'{CAMS[cam_idx]} — sampled pixels'); ax0.axis('off')
stride_y, stride_x = imH // fH, imW // fW
for (yy, xx) in sample_pixels:
    ax0.scatter([xx * stride_x + stride_x/2], [yy * stride_y + stride_y/2], s=120, edgecolor='white', linewidth=2)
    ax1.plot(depth_bins.cpu(), dd[:, yy, xx].cpu(), marker='o', label=f'pixel ({yy}, {xx})')
ax1.set_xlabel('depth [m]'); ax1.set_ylabel('P(depth)')
ax1.set_title('Categorical depth distribution at each marked pixel')
ax1.legend(); ax1.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 2.4 — The lifted 3D frustum

Each camera's feature map is *lifted* into a 3D frustum of shape `(D, fH, fW)`, one voxel per (depth, pixel) combination. Using the camera intrinsics and ego-frame extrinsics, `model.get_geometry(...)` returns the `(x, y, z)` location of every voxel in the ego frame.

We plot the top-magnitude voxels in 3D, colored by the weight of the lifted feature (higher = more confident the network placed features here).

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers 3d projection)

with torch.no_grad():
    geom = model.get_geometry(rots_b, trans_b, intrins_b, post_rots_b, post_trans_b)
# geom: (B, N, D, fH, fW, 3) in the ego frame
geom_cam = geom[0, cam_idx].cpu().numpy()                        # (D, fH, fW, 3)
weights  = lifted[cam_idx].norm(dim=0).cpu().numpy()             # (D, fH, fW)

# Keep only the top-magnitude voxels so the scatter is legible
thresh = np.quantile(weights, 0.90)
mask = weights > thresh
pts   = geom_cam[mask]
vals  = weights[mask]

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], c=vals, cmap='plasma', s=6, alpha=0.65)
ax.scatter([0], [0], [0], c='cyan', marker='^', s=120, edgecolor='black', label='ego')
ax.set_xlabel('x (forward) [m]'); ax.set_ylabel('y (left) [m]'); ax.set_zlabel('z [m]')
ax.set_title(f'{CAMS[cam_idx]} frustum AFTER the lift (top 10% voxels)')
ax.view_init(elev=22, azim=-70)
ax.legend(); plt.colorbar(sc, ax=ax, fraction=0.03, label='||lifted feature||')
plt.tight_layout(); plt.show()


### 2.5 — The **splat** — pooling all 6 cameras into one BEV grid

The final step is *splat*: every lifted voxel is scattered into its corresponding cell of a 2D BEV grid. Voxels that fall into the same cell are summed (the "cumulative-sum trick" in the LSS paper). This gives a single feature tensor `(C, 200, 200)` covering a 100 m × 100 m region around the ego vehicle.

We show two views of that tensor:
1. the L2 magnitude of the per-cell feature vector (where does LSS place *any* feature),
2. a PCA → RGB projection (what kind of features were placed where).

In [ ]:
with torch.no_grad():
    x_cam_feats = model.get_cam_feats(imgs_b)                    # (B, N, D, fH, fW, C)
    bev_feat    = model.voxel_pooling(geom, x_cam_feats)          # (B, C, X, Y)

bev_mag = bev_feat[0].norm(dim=0).cpu().numpy()                  # (X, Y)
bev_rgb = feat_to_rgb(bev_feat[0].cpu())                          # (X, Y, 3)

# to_topdown: transpose + flipud so that ego-X (forward) runs horizontally
# (right = forward) and ego-Y (left) runs vertically (up = left).
def to_topdown(arr):
    return np.flipud(arr.transpose(1, 0) if arr.ndim == 2 else arr.transpose(1, 0, 2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5.8))
extent = [-50, 50, -50, 50]
axes[0].imshow(to_topdown(bev_mag), cmap='magma', extent=extent)
axes[0].set_title('BEV feature magnitude ||f||₂')
axes[1].imshow(to_topdown(bev_rgb),                 extent=extent)
axes[1].set_title('BEV features  (PCA → RGB)')
for ax in axes:
    ax.scatter([0], [0], c='cyan', marker='^', s=180, edgecolor='black', label='ego')
    # Forward arrow: with this transposition, 'forward' points to the RIGHT of the plot.
    ax.annotate('', xy=(10, 0), xytext=(1.5, 0),
                arrowprops=dict(arrowstyle='-|>', color='cyan', lw=2.2, mutation_scale=18))
    ax.text(11.5, 0, 'forward', color='cyan', fontsize=9, va='center')
    ax.set_xlabel('x  [m]   (+ = forward)')      # horizontal = ego-X
    ax.set_ylabel('y  [m]   (+ = left of ego)')  # vertical   = ego-Y
    ax.set_xlim(-18, 18); ax.set_ylim(-18, 18)   # zoom in — feats are only ~15 m around the ego
    ax.set_aspect('equal')
    ax.legend(loc='upper right')
plt.suptitle('Splatted BEV features from all 6 cameras   (zoomed to ±18 m around the ego)', y=1.02)
plt.tight_layout(); plt.show()


**Recap of Part 2.** We walked through every stage of a Lift-Splat-Shoot forward pass:

| Stage | Tensor shape | What it means |
|------|--------------|---------------|
| 2D image features | `(N, C, fH, fW)` | per-pixel context descriptor |
| Depth distribution | `(N, D, fH, fW)` | `p(depth)` per pixel |
| Lifted 3D frustum | `(N, C, D, fH, fW)` | outer product `α ⊗ f` |
| Voxel-pooled BEV | `(1, C, 200, 200)` | everything summed into one BEV grid |

Up next — **Part 3**: using these same building blocks to look at depth *uncertainty* across the scene and compare it against ground truth when available. Then Parts 4–6 cover LiDAR-camera fusion, occupancy, and planning, all in the same BEV grid.

## Part 3 — **Shoot**: picking a trajectory on the BEV map

The *Shoot* in Lift-Splat-**Shoot** is the planning step: we have the drivable-area map from Part 1 and we want to pick the next trajectory for the ego vehicle.

Philion & Fidler do this (Section 5 of the paper) by:

1. Pre-mining a bank of **K template trajectories** from the nuScenes training data (k-means on observed ego motion).
2. At inference time, projecting each template onto the BEV drivable score and computing a **cost** along the trajectory.
3. Picking the template with the **lowest cost**.

We do the same thing with a small hand-crafted bank of 17 smooth lane-change candidates — the *mechanism* is identical.

> **Why not early fusion here?** Good question. LSS is camera-only — the pretrained weights (`model525000.pt`) have no LiDAR pathway. Adding a sparse-depth channel would require **retraining** the depth head with LiDAR supervision (that's BEVDepth / CaDDN). We'll save a proper BEV-fusion walkthrough for a **separate notebook** with weights from a fusion model.

### 3.1 — A bank of candidate trajectories

We sample **17 smooth paths** that all go 30 m forward but end at different lateral offsets, from −8 m (ego's right) to +8 m (ego's left). Each path starts at the ego with zero initial heading and uses a cubic-hermite blend (`3t² − 2t³`) for lateral displacement — so the ego never has to swerve instantly.

In [ ]:
from matplotlib.gridspec import GridSpec

# --- BEV display helper (forward up, ego-left on the left) ------------------
BEV_EXTENT = [-50, 50, -50, 50]
def bev_show(ax, arr, **kw):
    kw.setdefault('extent', BEV_EXTENT)
    im = ax.imshow(arr, origin='lower', **kw)
    ax.invert_xaxis()
    ax.set_xlabel('y  [m]   (+ = left of ego)')
    ax.set_ylabel('x  [m]   (+ = forward)')
    ax.set_aspect('equal')
    return im

def smooth_traj(x_end, y_end, length_pts=40):
    """Cubic-hermite-smoothed path from (0, 0) with zero initial heading to (x_end, y_end)."""
    t = np.linspace(0, 1, length_pts)
    x = x_end * t
    y = y_end * (3 * t**2 - 2 * t**3)
    return np.stack([x, y], axis=1)

LATERAL_TARGETS = np.linspace(-8.0, 8.0, 17)    # -8 m (right) … +8 m (left)
FORWARD_TARGET  = 30.0                          # metres ahead
trajectories = [smooth_traj(FORWARD_TARGET, lat) for lat in LATERAL_TARGETS]

# Quick preview on the LSS drivable score
fig, ax = plt.subplots(figsize=(7, 7))
bev_show(ax, bev, cmap='Blues', vmin=0, vmax=1, alpha=0.9)
for tr in trajectories:
    ax.plot(tr[:, 1], tr[:, 0], color='gray', alpha=0.55, lw=1.3)
ax.scatter([0], [0], c='cyan', marker='^', s=220, edgecolor='black', zorder=5, label='ego')
ax.set_title('3.1 — 17 candidate trajectories overlaid on the LSS drivable score',
             fontsize=11)
ax.set_xlim(28, -28); ax.set_ylim(-5, 45)
ax.legend(loc='lower left'); plt.tight_layout(); plt.show()


### 3.2 — Score each candidate against the BEV drivable map

For each candidate we convert its `(x, y)` waypoints to BEV cell indices, look up the drivable score at each cell, and compute the **cost** as the average of `1 − drivable_score` along the path.

- Staying on road  →  `drivable ≈ 1`  →  cost ≈ 0
- Going off-road   →  `drivable ≈ 0`  →  cost ≈ 1

In [ ]:
def trajectory_cost(traj, bev, grid_conf):
    xmin, _, xres = grid_conf['xbound']
    ymin, _, yres = grid_conf['ybound']
    xi = np.floor((traj[:, 0] - xmin) / xres).astype(int)
    yi = np.floor((traj[:, 1] - ymin) / yres).astype(int)
    ok = (xi >= 0) & (xi < bev.shape[0]) & (yi >= 0) & (yi < bev.shape[1])
    if not ok.any():
        return float('inf')
    xi, yi = np.clip(xi, 0, bev.shape[0]-1), np.clip(yi, 0, bev.shape[1]-1)
    drivable = bev[xi, yi]
    return float(np.mean(1.0 - drivable))

costs = np.array([trajectory_cost(tr, bev, grid_conf) for tr in trajectories])
best_idx = int(np.argmin(costs))
print(f'Best candidate: lateral target = {LATERAL_TARGETS[best_idx]:+.1f} m   cost = {costs[best_idx]:.3f}')

fig, ax = plt.subplots(figsize=(9, 3.3))
bars = ax.bar(LATERAL_TARGETS, costs, width=0.85, color='steelblue', edgecolor='black')
bars[best_idx].set_color('limegreen')
ax.set_xlabel('lateral target at 30 m  [m]   (+ = left of ego)')
ax.set_ylabel('cost  =  mean(1 − drivable)')
ax.set_title(f'3.2 — Cost for every candidate  (green bar = lowest-cost winner)')
ax.invert_xaxis()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


### 3.3 — The winning trajectory

We color-code all candidates by their cost (green = low, red = high), highlight the winner in lime, and — as a sanity check — project it back onto the **CAM_FRONT** image so you can visually verify it lands on actual road.

In [ ]:
# Project a trajectory (z = 0) into the CAM_FRONT image plane using
# the same intrinsics / extrinsics / post-aug transforms that LSS used.
def project_to_cam(ego_xy, cam_idx, batch, imH, imW):
    R  = batch[1][0, cam_idx].cpu().numpy()       # cam_to_ego rotation
    t  = batch[2][0, cam_idx].cpu().numpy()       # cam_to_ego translation
    K  = batch[3][0, cam_idx].cpu().numpy()       # camera intrinsics
    pr = batch[4][0, cam_idx].cpu().numpy()       # post-aug rotation (3x3)
    pt = batch[5][0, cam_idx].cpu().numpy()       # post-aug translation (3,)

    pts_ego = np.column_stack([ego_xy[:, 0], ego_xy[:, 1], np.zeros(len(ego_xy))])
    pts_cam = (pts_ego - t) @ R                   # ego -> cam  (R orthonormal, so R^-1 = R^T)
    keep = pts_cam[:, 2] > 0.1
    pts_cam = pts_cam[keep]
    if len(pts_cam) < 2:
        return np.array([]), np.array([])
    uv = pts_cam @ K.T
    u0, v0 = uv[:, 0] / uv[:, 2], uv[:, 1] / uv[:, 2]
    u = pr[0, 0] * u0 + pr[0, 1] * v0 + pt[0]
    v = pr[1, 0] * u0 + pr[1, 1] * v0 + pt[1]
    ok = (u >= 0) & (u < imW) & (v >= 0) & (v < imH)
    return u[ok], v[ok]

# --- side-by-side: BEV view (left) + CAM_FRONT projection (right) --------
norm_costs = (costs - costs.min()) / (costs.max() - costs.min() + 1e-8)
cmap_rgb   = plt.get_cmap('RdYlGn_r')

fig = plt.figure(figsize=(15, 6.8))
gs  = GridSpec(1, 2, width_ratios=[1, 1.35])

ax_bev = fig.add_subplot(gs[0])
bev_show(ax_bev, bev, cmap='Blues', alpha=0.85, vmin=0, vmax=1)
for i, tr in enumerate(trajectories):
    if i == best_idx: continue
    ax_bev.plot(tr[:, 1], tr[:, 0], color=cmap_rgb(norm_costs[i]), alpha=0.55, lw=1.8)
winner = trajectories[best_idx]
ax_bev.plot(winner[:, 1], winner[:, 0], color='lime',      lw=4.5, zorder=6, label=f'winner  ({LATERAL_TARGETS[best_idx]:+.1f} m)')
ax_bev.plot(winner[:, 1], winner[:, 0], color='darkgreen', lw=2.0, ls='--', zorder=7)
ax_bev.scatter([winner[-1, 1]], [winner[-1, 0]], c='lime', marker='*', s=350,
               edgecolor='black', zorder=8, label='goal')
ax_bev.scatter([0], [0], c='cyan', marker='^', s=220, edgecolor='black', zorder=5, label='ego')
ax_bev.set_title('BEV view — candidates colored by cost\n(green = low cost, red = high cost)',
                 fontsize=11)
ax_bev.set_xlim(28, -28); ax_bev.set_ylim(-5, 45)
ax_bev.legend(loc='lower left', fontsize=9)

ax_cam = fig.add_subplot(gs[1])
ax_cam.imshow(pil_images[CAMS.index('CAM_FRONT')])
u, v = project_to_cam(winner, CAMS.index('CAM_FRONT'), batch, imH, imW)
if len(u) >= 2:
    ax_cam.plot(u, v, '-', color='darkgreen', lw=5)
    ax_cam.plot(u, v, '-', color='lime',      lw=2.5)
    ax_cam.scatter(u[::4], v[::4], c='lime', s=70, edgecolor='black', zorder=5)
ax_cam.set_title('3.3 — Winning trajectory projected onto CAM_FRONT\n(sanity check: does it land on road?)',
                 fontsize=11)
ax_cam.axis('off')

fig.suptitle('Lift → Splat → **Shoot**  —  the pretrained LSS weights are enough to run the full pipeline on a single keyframe.',
             fontsize=12, y=1.03)
plt.tight_layout(); plt.show()


### Wrap-up of this notebook

| Stage | Tensor / object | What it means |
|------|-----------------|---------------|
| **Lift**  | `(N=6, C=64, D=41, 8, 22)` | every pixel feature ⊗ learned depth PDF |
| **Splat** | `(C=64, 200, 200)`         | everything pooled onto a shared BEV grid |
| **Shoot** | `(1, 1, 200, 200)` + trajectory bank | pick the lowest-cost path on the drivable map |

**Natural follow-up notebooks:**

1. **BEV Fusion lab** — swap LSS for BEVFusion or BEVDepth weights and compare camera-only vs camera-+-LiDAR predictions on the same keyframe. Early fusion (sparse-depth input channel) and late fusion (channel-concat on the BEV grid) can both be demonstrated cleanly once we have the right pretrained model.
2. **Occupancy lab** — load a model with an explicit 3-D occupancy head (FB-OCC, VoxFormer) and slice the predicted volume at multiple heights.
3. **Interactive depth-distribution explorer** — a click-to-inspect widget for the per-pixel depth PDF from section 2.3.